# 뉴스 데이터 xml 수집하기 - sbs news

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ✅ Session 재사용으로 HTTP 연결 효율화
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

# RSS 피드 수집
news_rss = session.get('https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07', timeout=10)
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')

# 제목과 링크를 한 번에 추출 (items 단위로 묶어서 처리)
items = news_rss_soup.select('item')
print(f"기사 개수: {len(items)}")

# ✅ zip 대신 item 단위로 title/link 동시 파싱
title_list = [item.select_one('title').text.strip() for item in items]
link_list  = [item.select_one('link').text.strip() for item in items]
print("제목 목록:", title_list)

# 각 기사 본문 수집
news_data = []
for rank, (title, link) in enumerate(zip(title_list, link_list), 1):
    try:
        news_response = session.get(link, timeout=10)
        news_response.raise_for_status()
        news_content_soup = BeautifulSoup(news_response.content, 'html.parser')
        news_content = news_content_soup.select_one("div[itemprop=articleBody]")

        # ✅ 본문 없을 경우 빈 문자열로 대체 (오류 방지)
        content_text = news_content.text.strip() if news_content else "본문 없음"
        news_data.append(content_text)
        print(f"[{rank}/{len(items)}] 수집 완료: {title[:20]}...")

    except Exception as e:
        news_data.append("수집 실패")
        print(f"[{rank}/{len(items)}] 오류 발생: {e}")

# ✅ DataFrame 생성 시 rank 컬럼 추가
news_df = pd.DataFrame(data={
    'rank'   : range(1, len(items) + 1),
    'title'  : title_list,
    'content': news_data
})
print(news_df.head())

news_df.to_csv("news.csv", encoding="utf-8-sig", index=False)
print("Save complete")

기사 개수: 29
제목 목록: ["빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트리 공개", '[바로이뉴스] "USA! USA!" 본격 \'미국 부흥회\'…"관세 위법" 대법원장 앞에선', "선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일]", '[속보] 트럼프, 집권 2기 첫 국정연설 시작', '군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵', '중국 드론 업체 DJI "증거 없이 수입 금지"…미국 법원에 소송', '미국, 인도·인도네시아 태양광 고율 관세…한화, 반사 이익?', '미국 최대 은행 JP모건도 AI발 대규모 인력 재배치 계획', '영국, 입국 전 전자여행허가 확대…한국 포함 85개국에 의무화', "'상호관세 돌려달라' 소송 봇물…로레알·다이슨도 합류", '멕시코 마약왕 사살 혼란에…혼다, 현지 공장 가동중단 뒤 재개', '술 먹는 침팬지…"야생 소변 샘플 20개 검사, 17개서 알코올 대사물 검출"', '미국, 유럽·중동에 군용기 150대 이동…이라크전 이후 최대', '미국, 서안지구서 첫 영사 서비스…이스라엘 "환영"', '이란, 미국과 핵협상 앞두고 "타결 가시권…외교 최우선해야"', '미국 법원, "오픈AI가 영업비밀 빼갔다" 주장한 xAI 소송 기각', '중국 누리꾼, "한국은 문화 도둑국" "중국설 훔쳤다" 주장', '길거리서 무료로…"참된 스승" 쏟아진 찬사', "스키 타다 '화들짝'…설원 가로질러 전력 질주", '노인 도왔더니 "4,500만 원 배상하라"…논란 터진 장면', '유엔총회, 우크라 지지 결의 채택…미·중은 기권', '오늘 국정연설…"백악관, 관세 15%로 인상 작업 중"', 'FBI국장이 왜 거기서 나와…미 하키팀 금메달 뒤풀이 참석 구설', '"미국, 은행에 고객 시민권정보 수집 요구 검토…이민단속 일환"', '보석 도둑맞은 루브르 박물관장 끝내 사임…마크롱 수락', '러 매체 "한국학자 란코프 국민대 교수, 라트비아서 체포"', '미국 유명앵커, 모친 실종 3주 만에 현상금 14억 내걸

In [ ]:
# print(news_response.url)
# print(news_response.status_code)
# print(news_content_soup.prettify()[:2000])  # 앞부분만 확인

# 뉴스 컨텐츠 클린징

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

# ① RSS 피드 수집
rss_response = session.get(
    'https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07', timeout=10
)
rss_response.raise_for_status()
soup = BeautifulSoup(rss_response.content, 'xml')

items = soup.select('item')
print(f"총 기사 개수: {len(items)}개\n")

# ② 제목 / 링크 / pubDate 추출
title_list = [item.select_one('title').text.strip()                                              for item in items]
link_list  = [item.select_one('link').text.strip()                                               for item in items]
date_list  = [item.select_one('pubDate').text.strip()[:16] if item.select_one('pubDate') else "날짜 없음"
                                                                                         for item in items]

# ③ 본문 클린징 함수
def fetch_and_clean(link):
    try:
        res = session.get(link, timeout=10)
        res.raise_for_status()
        body = BeautifulSoup(res.content, 'html.parser').select_one("div[itemprop=articleBody]")
        if body is None:
            return "본문 없음"
        text = re.sub(r'\s+', ' ', body.text.strip())
        text = re.sub(r'\[.*?\]', '', text)
        return text.strip()
    except Exception as e:
        print(f"  ⚠ 수집 실패: {link[:50]}... ({e})")
        return "본문 없음"

# ④ 기사별 본문 수집
news_data = []
for i, (title, link) in enumerate(zip(title_list, link_list), 1):
    print(f"[{i:02d}/{len(items)}] 수집 중: {title[:30]}...")
    news_data.append(fetch_and_clean(link))

# ⑤ DataFrame 생성
news_df = pd.DataFrame({
    'rank'           : range(1, len(items) + 1),
    'title'          : title_list,
    'pubDate'        : date_list,
    'content'        : news_data,
})
news_df['is_missing']     = news_df['content'] == "본문 없음"
news_df['content_length'] = news_df['content'].apply(lambda x: len(x) if x != "본문 없음" else 0)

# ⑥ 시각적으로 정리된 DataFrame 출력 (pandas 옵션만 사용)
pd.set_option('display.max_rows', 30)           # 최대 30행 표시
pd.set_option('display.max_columns', 10)        # 최대 10컬럼 표시
pd.set_option('display.width', 120)             # 출력 너비 120자
pd.set_option('display.max_colwidth', 35)       # 컬럼 내용 35자 제한 (넘으면 자동 ...)
pd.set_option('display.colheader_justify', 'center')  # 헤더 가운데 정렬

# 표시용 컬럼만 선택 + 한글 컬럼명으로 변경
display_df = news_df[['rank', 'title', 'pubDate', 'is_missing', 'content_length']].copy()
display_df.columns = ['순위', '제목', '날짜', '본문없음', '글자수']
display_df = display_df.set_index('순위')       # 순위를 인덱스로 → 왼쪽 기준 정렬 깔끔

print("\n[ SBS 뉴스 수집 결과 ]")
print(display_df.to_string())                   # 줄임 없이 전체 출력, 선 없이 깔끔하게

총 기사 개수: 29개

[01/29] 수집 중: 트럼프 집권 2기 첫 국정연설…"관세 정책 그대로"...
[02/29] 수집 중: 빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트...
[03/29] 수집 중: [바로이뉴스] "USA! USA!" 본격 '미국 부흥회...
[04/29] 수집 중: 선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [...
[05/29] 수집 중: 트럼프 "지금이 미 황금시대…우리의 적들은 두려워하고 ...
[06/29] 수집 중: 군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵...
[07/29] 수집 중: 중국 드론 업체 DJI "증거 없이 수입 금지"…미국 ...
[08/29] 수집 중: 미국, 인도·인도네시아 태양광 고율 관세…한화, 반사 ...
[09/29] 수집 중: 미국 최대 은행 JP모건도 AI발 대규모 인력 재배치 ...
[10/29] 수집 중: 영국, 입국 전 전자여행허가 확대…한국 포함 85개국에...
[11/29] 수집 중: '상호관세 돌려달라' 소송 봇물…로레알·다이슨도 합류...
[12/29] 수집 중: 멕시코 마약왕 사살 혼란에…혼다, 현지 공장 가동중단 ...
[13/29] 수집 중: 술 먹는 침팬지…"야생 소변 샘플 20개 검사, 17개...
[14/29] 수집 중: 미국, 유럽·중동에 군용기 150대 이동…이라크전 이후...
[15/29] 수집 중: 미국, 서안지구서 첫 영사 서비스…이스라엘 "환영"...
[16/29] 수집 중: 이란, 미국과 핵협상 앞두고 "타결 가시권…외교 최우선...
[17/29] 수집 중: 미국 법원, "오픈AI가 영업비밀 빼갔다" 주장한 xA...
[18/29] 수집 중: 중국 누리꾼, "한국은 문화 도둑국" "중국설 훔쳤다"...
[19/29] 수집 중: 길거리서 무료로…"참된 스승" 쏟아진 찬사...
[20/29] 수집 중: 스키 타다 '화들짝'…설원 가로질러 전력 질주...
[21/29] 수집 중: 노인 도왔더니 "4,500만 원 배상하라"…논란 터진

In [11]:
# ⑦ 수집 통계 요약 (표와 분리)
missing_count = news_df['is_missing'].sum()
total_count   = len(news_df)

print("\n" + "=" * 50)
print("[ 수집 통계 요약 ]")
print("=" * 50)
print(f"  본문 없음  : {missing_count}개")
print(f"  전체 기사  : {total_count}개")
print(f"  수집 성공률: {(total_count - missing_count) / total_count:.1%}")
print(f"  평균 본문  : {news_df['content_length'].mean():.0f}자")
print("=" * 50)


[ 수집 통계 요약 ]
  본문 없음  : 0개
  전체 기사  : 29개
  수집 성공률: 100.0%
  평균 본문  : 853자


In [12]:
# ⑧ CSV 저장
news_df.to_csv("news_clean.csv", encoding="utf-8-sig", index=False)
print("\nSave complete → news_clean.csv")


Save complete → news_clean.csv
